# Notebook 02 — Algoritmo Genético (Baseline Clásico)

**Reto:** Falcon Challenge — Resilient Release Scheduling  
**Equipo 04 — Hackathon OQI 2026**

## Descripción

Este notebook implementa la solución clásica al Falcon Challenge usando un **Algoritmo Genético (AG)**.  
El AG pertenece a la familia de algoritmos evolutivos — completamente distinta del Simulated Annealing (que es la base del Quantum Annealing) — y por eso es un baseline clásico válido y diferenciado.

## Problema a resolver

Dado el horizonte de planificación $T$ semanas con $L$ niveles de ajuste disponibles, encontrar la secuencia:

$$u^*(t) = \arg\max_{u(t)} \; SRS, \quad t = 0, 1, \ldots, T-1$$

donde:

$$SRS = -\left(w_1 C_{crit} + w_2 C_{dev} + w_3 C_{smooth}\right)$$

sujeto a las restricciones del benchmark oficial.

## Representación del cromosoma

Cada individuo es un vector de $T$ enteros, donde cada entero $\in \{0, 1, 2, 3, 4\}$ indica el índice del nivel de ajuste elegido en esa semana.  
Esto garantiza automáticamente que $u(t) \in \{-2\Delta u, -\Delta u, 0, \Delta u, 2\Delta u\}$ sin necesitar penalización one-hot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# ── Cargar datos procesados del Notebook 01 ─────────
R_obs_full       = np.load('R_obs_semanal.npy')
delta_S_obs_full = np.load('delta_S_obs_semanal.npy')
S0, Smax, Smin   = np.load('params_embalse.npy')

print(f'S0   = {S0:>20,.0f} m³')
print(f'Smax = {Smax:>20,.0f} m³')
print(f'Smin = {Smin:>20,.0f} m³')
print(f'Semanas disponibles: {len(R_obs_full)}')

In [ ]:
# ── Parámetros oficiales del benchmark ───────────────
T_OFICIAL = 26   # instancia mediana (benchmark oficial)
L         = 5    # niveles de ajuste
ETA       = 0.10 # restricción de balance

# Usar las primeras T semanas de datos
R_obs       = R_obs_full[:T_OFICIAL]
delta_S_obs = delta_S_obs_full[:T_OFICIAL]

# Escala de ajuste (Ecuación 10 del documento)
delta_u = 0.25 * np.median(R_obs)
umax    = 2 * delta_u
nivel_vals = np.array([-2, -1, 0, 1, 2]) * delta_u

# Pesos oficiales (Ecuación 12 del documento)
w1 = 1.0 / ((T_OFICIAL + 1) * Smin**2)
w2 = 0.1  / (T_OFICIAL * umax**2)
w3 = 0.1  / ((T_OFICIAL - 1) * (2*umax)**2)

print(f'delta_u    = {delta_u:>15,.0f} m³')
print(f'umax       = {umax:>15,.0f} m³')
print(f'nivel_vals = {nivel_vals}')
print(f'w1 = {w1:.4e}')
print(f'w2 = {w2:.4e}')
print(f'w3 = {w3:.4e}')

In [ ]:
# ── Funciones de simulación y evaluación ─────────────

def simular_storage(u_seq):
    """Calcula la trayectoria S(t) dada la secuencia de ajustes."""
    T = len(u_seq)
    S = np.zeros(T + 1)
    S[0] = S0
    for t in range(T):
        S[t+1] = S[t] + delta_S_obs[t] - u_seq[t]
    return S

def calcular_SRS(u_seq):
    """Calcula el Storage Resilience Score (SRS) oficial."""
    S       = simular_storage(u_seq)
    Ccrit   = np.sum(np.maximum(0, Smin - S)**2)
    Cdev    = np.sum(u_seq**2)
    Csmooth = np.sum(np.diff(u_seq)**2)
    return -(w1*Ccrit + w2*Cdev + w3*Csmooth)

def es_factible(u_seq):
    """
    Verifica las 4 restricciones físicas (fuera del QUBO):
    1. R(t) >= 0
    2. |u(t)| <= umax  (garantizada por construcción con nivel_vals)
    3. 0 <= S(t) <= Smax
    4. |sum(u(t))| <= eta * sum(R_obs)
    """
    if np.any(R_obs + u_seq < 0):
        return False
    if abs(np.sum(u_seq)) > ETA * np.sum(R_obs):
        return False
    S = simular_storage(u_seq)
    if np.any(S < 0) or np.any(S > Smax):
        return False
    return True

In [ ]:
# ── Baselines ─────────────────────────────────────────

# Baseline 1: histórico (u=0 siempre, reproduce operación observada)
u_hist  = np.zeros(T_OFICIAL)
S_hist  = simular_storage(u_hist)
SRS_hist = calcular_SRS(u_hist)

# Baseline 2: regla de umbral (Ecuación 18 del documento)
u_rule = np.zeros(T_OFICIAL)
S_temp = S0
for t in range(T_OFICIAL):
    u_rule[t] = -delta_u if S_temp < Smin else 0.0
    S_temp = S_temp + delta_S_obs[t] - u_rule[t]
S_rule  = simular_storage(u_rule)
SRS_rule = calcular_SRS(u_rule)

print(f'SRS histórico (u=0):   {SRS_hist:.6e}')
print(f'SRS baseline umbral:   {SRS_rule:.6e}')

In [ ]:
# ── Algoritmo Genético ────────────────────────────────

def fitness_ga(individuo):
    """
    Función de fitness para el AG.
    Usa penalización suave para restricciones físicas en lugar de rechazo duro,
    para preservar diversidad genética en la población.
    Los individuos infactibles tienen fitness reducido pero no eliminado.
    """
    u_seq = nivel_vals[individuo]
    S = np.zeros(T_OFICIAL + 1)
    S[0] = S0
    pen = 0.0

    for t in range(T_OFICIAL):
        S[t+1] = S[t] + delta_S_obs[t] - u_seq[t]
        if S[t+1] < 0:
            pen += abs(S[t+1]) * 1e-10
            S[t+1] = 0.0
        elif S[t+1] > Smax:
            pen += (S[t+1] - Smax) * 1e-10
            S[t+1] = Smax
        if R_obs[t] + u_seq[t] < 0:
            pen += abs(R_obs[t] + u_seq[t]) * 1e-10

    suma_u  = np.sum(u_seq)
    limite  = ETA * np.sum(R_obs)
    if abs(suma_u) > limite:
        pen += abs(abs(suma_u) - limite) * 1e-10

    Ccrit   = np.sum(np.maximum(0, Smin - S)**2)
    Cdev    = np.sum(u_seq**2)
    Csmooth = np.sum(np.diff(u_seq)**2)
    costo   = w1*Ccrit + w2*Cdev + w3*Csmooth + pen
    return -costo   # el AG maximiza, nosotros minimizamos costo


def algoritmo_genetico(
    tam_pob=200, n_gen=300, p_cruce=0.8,
    p_mut=0.15, elitismo=5, semilla=42
):
    """
    Algoritmo Genético para optimización de liberaciones de la Presa Falcon.

    Parámetros
    ----------
    tam_pob  : tamaño de la población
    n_gen    : número de generaciones
    p_cruce  : probabilidad de cruce entre padres
    p_mut    : probabilidad de mutación por individuo
    elitismo : número de mejores individuos preservados sin cambio

    Representación
    --------------
    Cromosoma: array de T_OFICIAL enteros ∈ {0,...,L-1}
    Gen t = índice del nivel de ajuste elegido en la semana t
    One-hot garantizado automáticamente: cada gen ya es un único índice.

    Operadores
    ----------
    Selección: torneo de tamaño 3
    Cruce    : un punto
    Mutación : cambio aleatorio de un gen a un nivel aleatorio
    """
    rng = np.random.default_rng(semilla)

    # Inicializar población aleatoriamente
    pob = rng.integers(0, L, size=(tam_pob, T_OFICIAL))
    # Insertar individuo histórico como semilla (u=0 para todo t)
    pob[0] = np.full(T_OFICIAL, L // 2)

    fit = np.array([fitness_ga(ind) for ind in pob])
    log = [fit.max()]

    for gen in range(n_gen):
        # Elitismo
        idx_elite = np.argsort(fit)[-elitismo:]
        elite     = pob[idx_elite].copy()

        # Generar nueva población
        nueva = []

        def torneo(k=3):
            ids = rng.integers(0, tam_pob, size=k)
            return pob[ids[np.argmax(fit[ids])]].copy()

        while len(nueva) < tam_pob - elitismo:
            p1, p2 = torneo(), torneo()

            # Cruce de un punto
            if rng.random() < p_cruce:
                pt = rng.integers(1, T_OFICIAL)
                h1 = np.concatenate([p1[:pt], p2[pt:]])
                h2 = np.concatenate([p2[:pt], p1[pt:]])
            else:
                h1, h2 = p1.copy(), p2.copy()

            # Mutación
            for h in [h1, h2]:
                if rng.random() < p_mut:
                    pos    = rng.integers(0, T_OFICIAL)
                    h[pos] = rng.integers(0, L)

            nueva.append(h1)
            if len(nueva) < tam_pob - elitismo:
                nueva.append(h2)

        pob = np.vstack([elite, np.array(nueva[:tam_pob - elitismo])])
        fit = np.array([fitness_ga(ind) for ind in pob])
        log.append(fit.max())

        if gen % 50 == 0:
            print(f'  Gen {gen:4d}/{n_gen} | Mejor fitness: {fit.max():.6e}')

    mejor_idx = np.argmax(fit)
    return pob[mejor_idx], fit[mejor_idx], log

In [ ]:
print('=== ALGORITMO GENÉTICO — T=26, L=5 ===')
t0 = time.time()
mejor_ind, mejor_fit, log_ga = algoritmo_genetico(
    tam_pob=200, n_gen=300, p_cruce=0.8, p_mut=0.15, elitismo=5
)
t_ga = time.time() - t0

u_ga = nivel_vals[mejor_ind]
S_ga = simular_storage(u_ga)
SRS_ga = calcular_SRS(u_ga)

print(f'\nTiempo de ejecución : {t_ga:.2f}s')
print(f'SRS GA              : {SRS_ga:.6e}')
print(f'¿Factible?          : {es_factible(u_ga)}')
print(f'\nΔSRS vs histórico   : {SRS_ga - SRS_hist:.6e}')
print(f'ΔSRS vs umbral      : {SRS_ga - SRS_rule:.6e}')

In [ ]:
# ── Visualización ─────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
sem = np.arange(T_OFICIAL + 1)

# Almacenamiento
axes[0].plot(sem, S_hist/1e9, '--', color='gray',   lw=2, label='Histórico (u=0)')
axes[0].plot(sem, S_rule/1e9, '-',  color='orange', lw=2, label='Baseline umbral')
axes[0].plot(sem, S_ga/1e9,   '-',  color='green',  lw=2.5, label='AG (clásico)')
axes[0].axhline(Smin/1e9, color='red', ls=':', lw=2, label=f'Smin')
axes[0].fill_between(sem, 0, Smin/1e9, alpha=0.07, color='red')
axes[0].set_title('Almacenamiento Presa Falcon — Algoritmo Genético', fontsize=13)
axes[0].set_ylabel('Almacenamiento (Gm³)')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Ajustes u(t)
semx = np.arange(T_OFICIAL)
axes[1].step(semx, u_ga/1e6, where='mid', color='green', lw=2, label='AG')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].axhline( umax/1e6, color='gray', ls=':', alpha=0.7, label=f'±umax')
axes[1].axhline(-umax/1e6, color='gray', ls=':', alpha=0.7)
axes[1].set_title('Ajustes de liberación u(t)', fontsize=13)
axes[1].set_ylabel('Ajuste (millones m³/semana)')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Convergencia
axes[2].plot(log_ga, color='green', lw=2)
axes[2].axhline(SRS_rule, color='orange', ls='--', label='Baseline umbral')
axes[2].set_title('Convergencia del Algoritmo Genético', fontsize=13)
axes[2].set_xlabel('Generación')
axes[2].set_ylabel('Mejor SRS')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('02_resultado_ga.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: 02_resultado_ga.png')

In [ ]:
# ── Guardar resultados para comparación final ─────────
np.save('u_ga.npy',         u_ga)
np.save('S_ga.npy',         S_ga)
np.save('u_hist.npy',       u_hist)
np.save('S_hist.npy',       S_hist)
np.save('u_rule.npy',       u_rule)
np.save('S_rule.npy',       S_rule)
np.save('tiempos_ga.npy',   np.array([t_ga]))
np.save('srs_ga.npy',       np.array([SRS_ga, SRS_hist, SRS_rule]))

print('Resultados guardados.')
print(f'\n=== RESUMEN ===')
print(f'SRS histórico  : {SRS_hist:.6e}')
print(f'SRS umbral     : {SRS_rule:.6e}')
print(f'SRS GA         : {SRS_ga:.6e}  ({t_ga:.2f}s)')